In [34]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [35]:
# Cost per delay minute by transit mode (CAD)
# Source: TTC Annual Report — operating cost per vehicle hour, divided by 60
# Streetcar is highest due to vehicle size and overhead wire infrastructure
COST_PER_MIN = {
    'Bus':       1.23,
    'Subway':    2.87,
    'Streetcar': 1.95,
}

# Prediction target
TARGET = 'Min Delay'

# Feature list — Min Gap and GapDeviation intentionally excluded
# Reason: gap size is partly caused by the delay itself (circular leakage)
# Removing them dropped R² from 0.95 to 0.27, confirming the leakage
FEATURES = [
    'Hour', 'Month', 'DayOfWeek',
    'IsRushHour', 'IsWeekend', 'IsNight',
    'Direction_known',
    'Season', 'CovidPhase', 'Incident', 'Vehicle', 'Route',
    'RouteAvgDelay', 'RouteDelayStd',
    'VehiclePrevDelay',
    'Temp', 'Precipitation', 'WindSpeed',
    'Humidity', 'WindChill', 'IsSnowCondition',
]

In [36]:
df_bus       = pd.read_csv('../data/clean_bus.csv')
df_sub       = pd.read_csv('../data/clean_sub.csv')
df_streetcar = pd.read_csv('../data/clean_streetcar.csv')

print(f'Bus rows      : {len(df_bus):,}')
print(f'Subway rows   : {len(df_sub):,}')
print(f'Streetcar rows: {len(df_streetcar):,}')

FileNotFoundError: [Errno 2] No such file or directory: '../data/clean_bus.csv'

In [27]:
import os
if not os.path.exists('data/weather_2023_2025.csv'):
    %run download_weather.py

In [6]:
def engineer_features(df):
    """
    Apply full feature engineering pipeline to a transit delay DataFrame.

    Steps:
        1. Normalize route column name (Line -> Route for subway/streetcar)
        2. Engineer time-based binary flags
        3. Assign season from month
        4. Assign COVID phase from year
        5. Join hourly weather data
        6. Compute route-level historical aggregates
        7. Compute vehicle lag feature (previous delay)

    Parameters
    ----------
    df : pd.DataFrame
        Cleaned transit delay dataframe with columns:
        Hour, DayOfWeek, Month, Year, Day, Route/Line, Vehicle, Min Delay, Min Gap

    Returns
    -------
    pd.DataFrame with all engineered features added
    """
    df = df.copy()

    # Normalize route column: subway and streetcar use 'Line', bus uses 'Route'
    if 'Line' in df.columns and 'Route' not in df.columns:
        df = df.rename(columns={'Line': 'Route'})

    # --- Time-based binary flags ---
    # Rush hour: AM peak (7-9) and PM peak (16-18)
    df['IsRushHour'] = np.where(
        df['Hour'].between(7, 9) | df['Hour'].between(16, 18), 1, 0
    )
    # Weekend: Saturday (5) and Sunday (6) — pandas dayofweek is 0=Monday
    df['IsWeekend'] = (df['DayOfWeek'] >= 5).astype(int)

    # Night service: midnight to 5am — lower supervision, different delay patterns
    df['IsNight'] = np.where(df['Hour'].between(0, 5), 1, 0)

    # --- Season from month ---
    # Toronto winter (Dec-Feb) is the highest-delay season for streetcar
    season_conditions = [
        df['Month'].isin([12, 1, 2]),   # Winter
        df['Month'].between(3, 5),       # Spring
        df['Month'].between(6, 8),       # Summer
        df['Month'].between(9, 11),      # Fall
    ]
    df['Season'] = np.select(
        season_conditions,
        ['Winter', 'Spring', 'Summer', 'Fall'],
        default='Unknown'
    )

    # --- COVID phase flag ---
    # Keeps all years in training but lets the model discount abnormal operating periods
    # 2020-2021: lockdown (20-40% of normal ridership)
    # 2022: recovery (gradual return to normal service levels)
    # 2023+: normal operations
    df['CovidPhase'] = 'Normal'
    df.loc[df['Year'] <= 2021, 'CovidPhase'] = 'Lockdown'
    df.loc[df['Year'] == 2022, 'CovidPhase'] = 'Recovery'

    # --- Join weather data ---
    # Left join on Year + Month + Day + Hour for hourly precision
    # Rows with no weather match (mostly COVID years) get median-filled below
    df = df.merge(weather, on=['Year', 'Month', 'Day', 'Hour'], how='left')

    # Fill unmatched weather rows
    for col in ['Precipitation', 'WindSpeed', 'IsSnowCondition']:
        df[col] = df[col].fillna(0)
    for col in ['Temp', 'Humidity', 'WindChill']:
        df[col] = df[col].fillna(df[col].median())

    # --- Route-level historical aggregates ---
    # Mean and std of delay per route — captures route-specific risk profile
    route_stats = df.groupby('Route')['Min Delay'].agg(
        RouteAvgDelay='mean',
        RouteDelayStd='std'
    ).reset_index()
    df = df.merge(route_stats, on='Route', how='left')

    # --- Gap deviation ---
    # How much this incident's gap differs from the route's average gap
    # NOTE: Min Gap itself is excluded from model features (leakage)
    # GapDeviation is also excluded but computed here for reference
    route_avg_gap      = df.groupby('Route')['Min Gap'].transform('mean')
    df['GapDeviation'] = df['Min Gap'] - route_avg_gap

    # --- Vehicle previous delay (lag feature) ---
    # Captures vehicle wear signal: a vehicle that was recently delayed is more likely to delay again
    # Sort by vehicle ID then chronological order before computing the shift
    df = df.sort_values(['Vehicle', 'Year', 'Month', 'Day', 'Hour'])
    df['VehiclePrevDelay'] = df.groupby('Vehicle')['Min Delay'].shift(1)

    # Fill NaN for each vehicle's first appearance with the route average
    df['VehiclePrevDelay'] = df['VehiclePrevDelay'].fillna(df['RouteAvgDelay'])

    return df

In [7]:
df_bus       = engineer_features(df_bus)
df_sub       = engineer_features(df_sub)
df_streetcar = engineer_features(df_streetcar)

print('Feature engineering complete.')
print(f'Bus columns      : {len(df_bus.columns)}')
print(f'Subway columns   : {len(df_sub.columns)}')
print(f'Streetcar columns: {len(df_streetcar.columns)}')

Feature engineering complete.
Bus columns      : 30
Subway columns   : 30
Streetcar columns: 30


In [8]:
def encode(df, encoders=None, fit=True):
    """
    Label-encode all object-type columns in a DataFrame.

    Parameters
    ----------
    df       : pd.DataFrame
    encoders : dict, optional
               Existing LabelEncoder objects. Pass during inference to reuse
               the same encoding as training.
    fit      : bool
               True  = fit new encoders (training)
               False = transform only using existing encoders (inference)

    Returns
    -------
    (pd.DataFrame, dict) — encoded DataFrame and encoder dictionary
    """
    df = df.copy()
    if encoders is None:
        encoders = {}

    for col in df.select_dtypes(include='object').columns:
        if col == TARGET:
            continue
        if fit:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            encoders[col] = le
        else:
            le = encoders[col]
            # Map unseen labels to -1 instead of raising ValueError
            df[col] = df[col].astype(str).map(
                lambda x: le.transform([x])[0] if x in le.classes_ else -1
            )

    return df, encoders

In [9]:
datasets = {
    'Bus':       df_bus,
    'Subway':    df_sub,
    'Streetcar': df_streetcar,
}

results = {}

for name, df in datasets.items():
    print(f"\n{'='*54}")
    print(f'  {name}')
    print(f"{'='*54}")

    # Keep only features that exist in this dataframe
    # (some features may differ slightly between modes)
    features = [f for f in FEATURES if f in df.columns]

    # Drop rows with any null in features or target
    data = df[features + [TARGET, 'Year']].dropna()

    # Time-based train/test split
    # Training: all data up to and including 2023
    # Testing: 2024 only (unseen future data)
    train = data[data['Year'] <= 2023]
    test  = data[data['Year'] == 2024]

    if len(test) == 0:
        print(f'  No 2024 data found — skipping {name}')
        continue

    # Encode categoricals — fit on train, transform test with same encoders
    train_enc, encoders = encode(train[features + [TARGET]], fit=True)
    test_enc,  _        = encode(test[features + [TARGET]],  fit=False, encoders=encoders)

    X_train = train_enc[features]
    y_train = train_enc[TARGET]
    X_test  = test_enc[features]
    y_test  = test_enc[TARGET]

    # Gradient Boosting Regressor
    # n_estimators=300: enough trees for stable predictions without overfitting
    # learning_rate=0.05: conservative step size, pairs with high n_estimators
    # subsample=0.8: row sampling per tree — reduces variance
    # min_samples_leaf=10: prevents learning from very rare delay patterns
    model = GradientBoostingRegressor(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        min_samples_leaf=10,
        random_state=42,
    )
    model.fit(X_train, y_train)

    # Clip negative predictions — delay cannot be less than 0 minutes
    y_pred = np.clip(model.predict(X_test), 0, None)

    # Evaluation metrics
    r2   = r2_score(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    # Cost estimation
    cost_per_min         = COST_PER_MIN[name]
    avg_predicted_cost   = y_pred.mean() * cost_per_min
    total_predicted_cost = y_pred.sum()  * cost_per_min

    print(f'  Rows (train): {len(X_train):,}  |  Rows (test): {len(X_test):,}')
    print(f'  R2  : {r2:.4f}  (note: low R2 is expected for transit delay data — see notebook header)')
    print(f'  MAE : {mae:.2f} min')
    print(f'  RMSE: {rmse:.2f} min')
    print(f'\n  Cost per minute  : ${cost_per_min:.2f} CAD')
    print(f'  Avg cost/incident: ${avg_predicted_cost:.2f} CAD')
    print(f'  Total est. cost  : ${total_predicted_cost:,.0f} CAD  (2024 test period)')

    # Feature importances — top 10 by contribution to prediction variance
    fi    = pd.Series(model.feature_importances_, index=features)
    top10 = fi.sort_values(ascending=False).head(10)
    print('\n  Top 10 Feature Importances:')
    for feat, imp in top10.items():
        print(f'    {feat:<22} {imp:.4f}')

    # Store results for later use (summary table + inference)
    results[name] = {
        'model':                 model,
        'encoders':              encoders,
        'features':              features,
        'r2':                    r2,
        'mae':                   mae,
        'rmse':                  rmse,
        'avg_cost_per_incident': avg_predicted_cost,
        'total_cost':            total_predicted_cost,
        'y_test':                y_test,
        'y_pred':                y_pred,
    }


  Bus
  Rows (train): 112,619  |  Rows (test): 56,630
  R2  : 0.2778  (note: low R2 is expected for transit delay data — see notebook header)
  MAE : 11.77 min
  RMSE: 42.86 min

  Cost per minute  : $1.23 CAD
  Avg cost/incident: $24.26 CAD
  Total est. cost  : $1,373,563 CAD  (2024 test period)

  Top 10 Feature Importances:
    Incident               0.6013
    RouteDelayStd          0.0908
    RouteAvgDelay          0.0600
    Hour                   0.0492
    VehiclePrevDelay       0.0434
    Vehicle                0.0383
    CovidPhase             0.0181
    Month                  0.0175
    WindChill              0.0171
    Humidity               0.0129

  Subway
  Rows (train): 43,235  |  Rows (test): 26,307
  R2  : 0.1595  (note: low R2 is expected for transit delay data — see notebook header)
  MAE : 2.47 min
  RMSE: 9.27 min

  Cost per minute  : $2.87 CAD
  Avg cost/incident: $8.27 CAD
  Total est. cost  : $217,606 CAD  (2024 test period)

  Top 10 Feature Importances:
   

In [10]:
summary_rows = []
for name, v in results.items():
    summary_rows.append({
        'Mode':              name,
        'R2':                round(v['r2'],   4),
        'MAE (min)':         round(v['mae'],  2),
        'RMSE (min)':        round(v['rmse'], 2),
        'Avg Cost/Incident': f"${v['avg_cost_per_incident']:.2f} CAD",
        'Total Cost (2024)': f"${v['total_cost']:,.0f} CAD",
    })

summary = pd.DataFrame(summary_rows).set_index('Mode')
summary

,R2,MAE (min),RMSE (min),Avg Cost/Incident,Total Cost (2024)
Mode,,,,,
Bus,0.2778,11.77,42.86,$24.26 CAD,"$1,373,563 CAD"
Subway,0.1595,2.47,9.27,$8.27 CAD,"$217,606 CAD"
Streetcar,0.0582,12.97,34.07,$33.10 CAD,"$464,727 CAD"


In [11]:
def predict_delay_cost(mode, new_df):
    """
    Predict delay duration and estimated cost for new incident rows.

    Parameters
    ----------
    mode   : str
             One of 'Bus', 'Subway', 'Streetcar'
    new_df : pd.DataFrame
             Must contain the same feature columns used in training.
             Weather columns (Temp, Precipitation, etc.) should be joined in beforehand.
             CovidPhase defaults to 'Normal' if not provided.

    Returns
    -------
    pd.DataFrame
        Input DataFrame with two additional columns:
        - predicted_delay_min : float — predicted delay in minutes
        - predicted_cost_cad  : float — estimated economic cost in CAD

    Example
    -------
    new_row = pd.DataFrame([{
        'Hour': 8, 'Month': 1, 'DayOfWeek': 0,
        'Direction_known': 1, 'Season': 'Winter', 'CovidPhase': 'Normal',
        'Incident': 'Mechanical', 'Vehicle': 8001, 'Route': 29,
        'RouteAvgDelay': 4.2, 'RouteDelayStd': 3.1, 'VehiclePrevDelay': 3.0,
        'Temp': -5.0, 'Precipitation': 0.0, 'WindSpeed': 20.0,
        'Humidity': 80.0, 'WindChill': -12.0, 'IsSnowCondition': 0,
    }])
    result = predict_delay_cost('Bus', new_row)
    print(result[['predicted_delay_min', 'predicted_cost_cad']])
    """
    if mode not in results:
        raise ValueError(f"Mode '{mode}' not found. Run the training cell first.")

    r        = results[mode]
    features = r['features']
    encoders = r['encoders']
    model    = r['model']

    df = new_df.copy()

    # Default CovidPhase to Normal — future predictions are post-pandemic
    if 'CovidPhase' not in df.columns:
        df['CovidPhase'] = 'Normal'

    # Auto-compute binary flags if not already present
    if 'IsRushHour' not in df.columns:
        df['IsRushHour'] = np.where(
            df['Hour'].between(7, 9) | df['Hour'].between(16, 18), 1, 0
        )
    if 'IsWeekend' not in df.columns:
        df['IsWeekend'] = (df['DayOfWeek'] >= 5).astype(int)
    if 'IsNight' not in df.columns:
        df['IsNight'] = np.where(df['Hour'].between(0, 5), 1, 0)
    if 'IsSnowCondition' not in df.columns and 'Temp' in df.columns:
        df['IsSnowCondition'] = (
            (df['Temp'] <= 2) & (df['Precipitation'] > 0)
        ).astype(int)

    # Encode using the same encoders fitted during training
    df, _ = encode(df, encoders=encoders, fit=False)

    # Predict and clip negatives
    preds = np.clip(model.predict(df[features]), 0, None)

    df['predicted_delay_min'] = preds
    df['predicted_cost_cad']  = preds * COST_PER_MIN[mode]

    return df

In [12]:
# Example: predict cost of a mechanical incident on Bus Route 29
# during morning rush hour on a cold winter weekday
example_incident = pd.DataFrame([{
    'Hour':            8,
    'Month':           1,
    'DayOfWeek':       0,       # Monday
    'Direction_known': 1,
    'Season':          'Winter',
    'CovidPhase':      'Normal',
    'Incident':        'Mechanical',
    'Vehicle':         8001,
    'Route':           29,
    'RouteAvgDelay':   4.2,
    'RouteDelayStd':   3.1,
    'VehiclePrevDelay': 3.0,
    'Temp':            -8.0,
    'Precipitation':    0.0,
    'WindSpeed':       25.0,
    'Humidity':        72.0,
    'WindChill':       -15.0,
    'IsSnowCondition':  0,
}])

prediction = predict_delay_cost('Bus', example_incident)
print(f"Predicted delay : {prediction['predicted_delay_min'].values[0]:.1f} minutes")
print(f"Estimated cost  : ${prediction['predicted_cost_cad'].values[0]:.2f} CAD")

Predicted delay : 3.1 minutes
Estimated cost  : $3.78 CAD
